In [1]:
import pandas as pd
from pathlib import Path

OUT = Path("../data_processed")
PROMPTS = Path("../prompts")

products = pd.read_csv(OUT / "product_facts.csv")
reviews = pd.read_csv(OUT / "reviews.csv")

# 关键词候选（你 Day3 导出的 candidates）
lex_pos = pd.read_csv(OUT / "keyword_lexicon_pos_candidates.csv")
lex_neg = pd.read_csv(OUT / "keyword_lexicon_neg_candidates.csv")

# 做成 {category: [keywords]} 映射（取前 20 个）
pos_map = (lex_pos.sort_values(["category","freq"], ascending=[True, False])
           .groupby("category")["keyword"].apply(lambda x: list(x.head(20))).to_dict())
neg_map = (lex_neg.sort_values(["category","freq"], ascending=[True, False])
           .groupby("category")["keyword"].apply(lambda x: list(x.head(20))).to_dict())

def build_fact_sheet(row):
    # 只允许使用这些字段（事实表）
    lines = []
    lines.append(f"product_id: {row.get('product_id','')}")
    lines.append(f"brand_name: {row.get('brand_name','')}")
    lines.append(f"product_name: {row.get('product_name','')}")
    lines.append(f"category: {row.get('primary_category','')} / {row.get('secondary_category','')} / {row.get('tertiary_category','')}")
    lines.append(f"price_usd: {row.get('price_usd','')}")
    lines.append(f"sale_price_usd: {row.get('sale_price_usd','')}")
    lines.append(f"rating_avg: {row.get('rating_avg','')}")
    lines.append(f"review_cnt: {row.get('review_cnt','')}")
    lines.append(f"ingredients: {row.get('ingredients','')}")
    lines.append(f"highlights: {row.get('highlights','')}")
    return "\n".join(lines)

# 选一个示例产品（你也可以换成指定 product_id）
example = products.sample(1, random_state=42).iloc[0]
fact_sheet = build_fact_sheet(example)

category = example["primary_category"]
keywords = pos_map.get(category, [])
risk_terms = neg_map.get(category, [])

print("CATEGORY:", category)
print("\n--- FACT_SHEET ---\n", fact_sheet)
print("\n--- KEYWORDS ---\n", keywords[:10])
print("\n--- RISK_TERMS ---\n", risk_terms[:10])

CATEGORY: Hair

--- FACT_SHEET ---
 product_id: P448733
brand_name: Virtue
product_name: Hydrating Recovery Shampoo for Dry, Damaged & Colored Hair
category: Hair / Shampoo & Conditioner / Shampoo
price_usd: 40.0
sale_price_usd: 40.0
rating_avg: 4.183
review_cnt: 0
ingredients: nan
highlights: ['Good for: Damage', 'Vegan', 'Good for: Color Care', 'Without Parabens', 'Without Sulfates SLS & SLES']

--- KEYWORDS ---
 []

--- RISK_TERMS ---
 []


In [2]:
def render_template(path: Path, **kwargs):
    t = path.read_text(encoding="utf-8")
    for k, v in kwargs.items():
        t = t.replace("{"+k+"}", v if isinstance(v, str) else str(v))
    return t

title_prompt = render_template(
    PROMPTS / "title.txt",
    FACT_SHEET=fact_sheet,
    KEYWORDS=", ".join(keywords[:10])
)

print(title_prompt[:1200])

You are an e-commerce copywriter.

STRICT FACT RULE:
- You may ONLY use facts from FACT_SHEET below.
- If a fact is not present in FACT_SHEET, write "未明确说明" or omit it.
- Do NOT invent ingredients, claims, certifications, effects, medical benefits, or target groups.

TASK:
Generate ONE product title.

OUTPUT CONSTRAINTS:
- <= 60 English characters (or <= 30 Chinese characters)
- Must include: brand_name OR a short part of product_name
- Must include: 1-2 category keywords from KEYWORDS (if available)

FACT_SHEET:
product_id: P448733
brand_name: Virtue
product_name: Hydrating Recovery Shampoo for Dry, Damaged & Colored Hair
category: Hair / Shampoo & Conditioner / Shampoo
price_usd: 40.0
sale_price_usd: 40.0
rating_avg: 4.183
review_cnt: 0
ingredients: nan
highlights: ['Good for: Damage', 'Vegan', 'Good for: Color Care', 'Without Parabens', 'Without Sulfates SLS & SLES']

KEYWORDS (category high-frequency terms):


Return ONLY the title text.

